# Sales JSON - Gold analytics

**Medallion role:** Gold. This notebook turns validated Silver orders into business-facing daily, category, and customer summaries. Gold tables contain reusable metrics at explicit grains so reporting tools do not need to scan or reinterpret order-level data.

In [0]:
dbutils.widgets.removeAll()

## Analytical contract

The environment widget resolves the Silver source and external Gold targets. Each target has its own business key and storage path, which makes refresh behavior and ownership clear across environments.

In [0]:
from datetime import datetime, timezone

dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

dbutils.widgets.text(
    "ingestion_timestamp",
    datetime.now(timezone.utc).isoformat(),
    "Ingestion Timestamp"
)

environment = dbutils.widgets.get("environment").lower()

ingestion_timestamp = (
    dbutils.widgets.get("ingestion_timestamp").strip()
)

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "storage_account": "stcentralusjrdev",
        "catalog": "salesjson_dev"
    },
    "prod": {
        "storage_account": "stcentralusjrprod",
        "catalog": "salesjson_prod"
    }
}

env = config[environment]

storage_account = env["storage_account"]
catalog = env["catalog"]

silver_table = f"{catalog}.silver.orders"

daily_table = (
    f"{catalog}.gold.daily_sales_summary"
)

category_table = (
    f"{catalog}.gold.category_sales_summary"
)

customer_table = (
    f"{catalog}.gold.customer_sales_summary"
)

daily_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salesjson/gold/daily_sales_summary/"
)

category_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salesjson/gold/category_sales_summary/"
)

customer_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salesjson/gold/customer_sales_summary/"
)

print(f"Environment         : {environment}")
print(f"Source              : {silver_table}")
print(f"Daily target        : {daily_table}")
print(f"Category target     : {category_table}")
print(f"Customer target     : {customer_table}")

In [0]:
silver_df = spark.table(silver_table)

## Curated business aggregates

The following transformations calculate order volume, customer reach, units, revenue, discounts, and order-value statistics at daily, category, and customer grains. All measures originate from the cleansed Silver table, preserving the Medallion dependency chain.

In [0]:
from pyspark.sql.functions import (
    col,
    to_date,
    countDistinct,
    sum,
    avg,
    min,
    max,
    round,
    lit
)

daily_sales_df = (
    silver_df

    .withColumn(
        "sale_date",
        to_date(col("order_timestamp"))
    )

    .groupBy("sale_date")

    .agg(
        countDistinct("order_id")
            .alias("total_orders"),

        countDistinct("customer_id")
            .alias("total_customers"),

        sum("quantity")
            .alias("total_units"),

        round(
            sum("gross_amount"),
            2
        ).alias("gross_revenue"),

        round(
            sum("discount_amount"),
            2
        ).alias("total_discount"),

        round(
            sum("net_amount"),
            2
        ).alias("net_revenue"),

        round(
            avg("net_amount"),
            2
        ).alias("average_order_value"),

        round(
            min("net_amount"),
            2
        ).alias("min_order_value"),

        round(
            max("net_amount"),
            2
        ).alias("max_order_value")
    )

    .withColumn(
        "gold_processing_timestamp",
        lit(ingestion_timestamp).cast("timestamp")
    )
)

In [0]:
category_sales_df = (
    silver_df

    .groupBy("category")

    .agg(
        countDistinct("order_id")
            .alias("total_orders"),

        countDistinct("customer_id")
            .alias("total_customers"),

        sum("quantity")
            .alias("total_units"),

        round(
            sum("gross_amount"),
            2
        ).alias("gross_revenue"),

        round(
            sum("discount_amount"),
            2
        ).alias("total_discount"),

        round(
            sum("net_amount"),
            2
        ).alias("net_revenue"),

        round(
            avg("net_amount"),
            2
        ).alias("average_order_value"),

        round(
            min("net_amount"),
            2
        ).alias("min_order_value"),

        round(
            max("net_amount"),
            2
        ).alias("max_order_value")
    )

    # PySpark equivalent of applying HAVING
    # after an aggregation.
    .filter(
        col("total_orders") >= 2
    )

    .withColumn(
        "gold_processing_timestamp",
        lit(ingestion_timestamp).cast("timestamp")
    )
)

In [0]:
customer_sales_df = (
    silver_df

    .groupBy(
        "customer_id",
        "country"
    )

    .agg(
        countDistinct("order_id")
            .alias("total_orders"),

        sum("quantity")
            .alias("total_units"),

        round(
            sum("gross_amount"),
            2
        ).alias("gross_revenue"),

        round(
            sum("discount_amount"),
            2
        ).alias("total_discount"),

        round(
            sum("net_amount"),
            2
        ).alias("total_spent"),

        round(
            avg("net_amount"),
            2
        ).alias("average_order_value"),

        min("order_timestamp")
            .alias("first_order_timestamp"),

        max("order_timestamp")
            .alias("last_order_timestamp")
    )

    .withColumn(
        "gold_processing_timestamp",
        lit(ingestion_timestamp).cast("timestamp")
    )
)

## Repeatable Gold refreshes

Each external Delta target is created on its first run and MERGEd thereafter using its declared grain. Matched rows are refreshed, new groups are inserted, and groups no longer present in the current Silver snapshot are removed so Gold remains a synchronized analytical snapshot.

In [0]:
from delta.tables import DeltaTable


def merge_gold_snapshot(
    source_df,
    target_table,
    target_path,
    merge_condition
):
    """
    Creates an external Gold Delta table during the initial load.

    Subsequent executions synchronize the Gold table with the
    latest aggregate snapshot using Delta MERGE.
    """

    if not spark.catalog.tableExists(target_table):

        print(
            f"Initial load. Creating Gold table: "
            f"{target_table}"
        )

        (
            source_df.write
                .format("delta")
                .mode("append")
                .option("mergeSchema", "true")
                .option("path", target_path)
                .saveAsTable(target_table)
        )

    else:

        print(
            f"Synchronizing Gold table: "
            f"{target_table}"
        )

        target = DeltaTable.forName(
            spark,
            target_table
        )

        (
            target.alias("target")

                .merge(
                    source_df.alias("source"),
                    merge_condition
                )

                .withSchemaEvolution()

                .whenMatchedUpdateAll()

                .whenNotMatchedInsertAll()

                .whenNotMatchedBySourceDelete()

                .execute()
        )

    print(
        f"Gold synchronization completed: "
        f"{target_table}"
    )

In [0]:
merge_gold_snapshot(
    source_df=daily_sales_df,
    target_table=daily_table,
    target_path=daily_path,
    merge_condition=(
        "target.sale_date = source.sale_date"
    )
)

In [0]:
merge_gold_snapshot(
    source_df=category_sales_df,
    target_table=category_table,
    target_path=category_path,
    merge_condition=(
        "target.category = source.category"
    )
)

In [0]:
# COMMAND ----------

merge_gold_snapshot(
    source_df=customer_sales_df,
    target_table=customer_table,
    target_path=customer_path,
    merge_condition=(
        "target.customer_id = source.customer_id "
        "AND target.country = source.country"
    )
)